# log-samples-eval-callback — ex1: every-K-steps eval callback that logs N samples to a sink

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `log-samples-eval-callback`. Running the final beacon cell reports progress against the `Logging: log-samples eval callback` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Logging: log-samples eval callback` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`log-samples-eval-callback`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "log-samples-eval-callback"
DD_SUBTOPIC = "Logging: log-samples eval callback"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Every-K-steps eval callback — quick refresher

Many ARENA training loops periodically dump a small batch of model outputs to a logging sink — sample completions from a language model, sample generations from a diffusion model, eval-set predictions, etc.

**The pattern:**

```python
for step, batch in enumerate(loader):
    loss = train_step(batch)
    if step % args.eval_every == 0:
        samples = sample_n_from_model(model, n=args.n_eval)
        sink.append({'step': step, 'samples': samples})
```

**Three knobs:**

- `eval_every` (K) — how often to fire the callback. K=100 is common for cheap eval, K=1000 for expensive generation.
- `n_eval` (N) — how many samples to log per fire. Keep small (N≤16) to avoid blowing up wandb storage / slowing training.
- `sink` — where the samples go. Can be `wandb.log({'samples': ...})`, a `wandb.Table`, or a plain Python list (for offline debugging).

**The plain-list sink** is what this drill uses — it lets you exercise the every-K cadence + N-per-fire counting WITHOUT requiring wandb to be installed. The cadence logic is identical whatever sink you wire in.

### Exercise 1 — every-K-steps eval callback that logs N samples to a sink

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply the every-K-steps callback pattern to dump N deterministically-generated model outputs to a plain-list sink, verified by checking cadence + per-fire sample count.
> Keywords: eval, callback, cadence, sink
> ```

**KCs targeted:** `modulo-k-step-cadence`, `n-sample-dump-to-sink`

Implement `ex1_run_with_sample_callback(n_steps, eval_every, n_eval, sink)`. The canonical 'log samples every K steps' training-loop pattern (with a plain-list sink so we don't need wandb):

1. Loop `for step in range(n_steps)`.
2. At every step where `step % eval_every == 0` (so step 0, K, 2K, ...):
   - Generate `n_eval` fake samples — use `[f'step={step}-sample={i}' for i in range(n_eval)]`.
   - Append `{'step': step, 'samples': samples}` to `sink`.
3. Return the total number of `sink.append` calls.

**The sink is a plain Python list.** No wandb dependency. In real ARENA code you'd swap `sink.append(d)` for `wandb.log(d, step=step)` — same cadence logic, different destination.

**Step 0 fires.** `0 % K == 0` for any K, so the initial random model gets a baseline sample dump before training begins. This is the ARENA convention.

In [ ]:
def ex1_run_with_sample_callback(n_steps, eval_every, n_eval, sink):
    n_fires = 0
    for step in range(n_steps):
        if step % eval_every == 0:
            samples = [f'step={step}-sample={i}' for i in range(n_eval)]
            sink.append({'step': step, 'samples': samples})
            n_fires += 1
    return n_fires


<details><summary>Solution</summary>

```python
def ex1_run_with_sample_callback(n_steps, eval_every, n_eval, sink):
    n_fires = 0
    for step in range(n_steps):
        if step % eval_every == 0:
            samples = [f'step={step}-sample={i}' for i in range(n_eval)]
            sink.append({'step': step, 'samples': samples})
            n_fires += 1
    return n_fires
```

**`step % K == 0` includes step 0.** The first iteration of the loop, BEFORE any training has happened, captures the random-init baseline. Skip step 0 if you specifically want trained-only samples — but ARENA's convention is to include it.

**Why a plain list sink.** In ARENA's real loop you'd write `wandb.log({'samples': wandb.Table(data=...)}, step=step)` here. By swapping in a list we exercise the SAME cadence logic (`step % K == 0`) and the SAME N-per-fire counting without needing wandb installed. The cadence is the load-bearing primitive; the destination is interchangeable.

**`n_steps=0` is a legitimate edge case.** Tests that construct a model and immediately exit (e.g. smoke tests) should produce zero samples, not crash.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()